In [ ]:
prompt="Change the background of the image"
url = "c:\downloads\others\overweight-man-in-his-sixties-out-in-the-fresh-air.webp"

In [12]:
import requests
import base64
from IPython.display import Image, display
import os
from dotenv import load_dotenv

load_dotenv

# 1. Setup Client
REPLICATE_TOKEN = os.getenv("REPLICATE_API_TOKEN")

# ────────────────────────────────────────────────         # ← your token
prompt = "change the background to a starry night sky, keep the person exactly the same, realistic"
input_image_path = "c:\downloads\others\overweight-man-in-his-sixties-out-in-the-fresh-air.webp"            # or URL
# ────────────────────────────────────────────────

# Read and encode image (local file)
with open(input_image_path, "rb") as f:
    image_base64 = base64.b64encode(f.read()).decode("utf-8")

headers = {
    "Authorization": f"Bearer {REPLICATE_API_TOKEN}",
    "Content-Type": "application/json"
}

payload = {
    "version": "prunaai/firered-image-edit",   # latest version at time of writing
    "input": {
        "image": f"data:image/jpeg;base64,{image_base64}",
        "prompt": prompt,
        "num_inference_steps": 40,
        "guidance_scale": 7.0,
        # optional: "negative_prompt": "blurry, deformed, low quality",
        # "seed": 42
    }
}

response = requests.post(
    "https://api.replicate.com/v1/predictions",
    headers=headers,
    json=payload
)

if response.status_code != 201:
    print("Error:", response.json())
else:
    pred = response.json()
    print("Prediction started → polling status...")
    
    # Poll until done (simple version)
    while True:
        status_resp = requests.get(pred["urls"]["get"], headers=headers)
        status = status_resp.json()
        
        if status["status"] == "succeeded":
            output_url = status["output"]
            print("Done! Output:", output_url)
            
            # Display in notebook
            img_data = requests.get(output_url).content
            display(Image(data=img_data))
            break
            
        elif status["status"] in ["failed", "canceled"]:
            print("Failed:", status.get("error", "Unknown error"))
            break
            
        print("Waiting...")
        import time
        time.sleep(3)

Error: {'title': 'Unauthenticated', 'detail': 'You did not pass a valid authentication token', 'status': 401}
